# Project 1: News Scraper dengan BeautifulSoup
**Target:** Scraping berita dari antaranews.com (RSS Feed)

**Tech Stack:** `requests`, `BeautifulSoup`, `pandas`, `feedparser`

**Use Case:** Mengambil judul, link, tanggal, dan ringkasan berita terkini secara otomatis.


BeautifulSoup adalah library Python yang digunakan untuk parsing HTML dan XML, sehingga memudahkan proses ekstraksi data dari halaman web.

Dalam project ini, BeuagtifulSoup digunakan untuk:


*   Membersihkan tag HTML pada ringkasan berita dari RSS feed
*   Mengambil isi artikel lengkap dari halaman web
*   Mengubah struktur HTML menjadi teks yang dapat dianalisis



## 1. Install & Import Library

In [1]:
!pip install requests beautifulsoup4 pandas feedparser lxml -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 5.1 MB/s eta 0:00:00


In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import feedparser
from datetime import datetime
import time
import re

#2. Konfigurasi Target Scraping

In [3]:
# Daftar RSS Feed dari berbagai kategori antaranews
RSS_FEEDS = {
    'terkini': 'https://www.antaranews.com/rss/terkini.xml',
    'ekonomi': 'https://www.antaranews.com/rss/ekonomi.xml',
}

#Menghindari pemblokiran dari server
#Digunakan agar request terlihat seperti browser biasa
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36'
}

print(f'📡 Target: {len(RSS_FEEDS)} kategori berita')

📡 Target: 2 kategori berita


#3. Scaper Class

In [17]:
class NewsScraper:
    def __init__(self, headers=None):
        self.headers = headers or {}
        self.all_articles = []

    def scrape_rss(self, category, url):
        """Scrape berita dari RSS feed."""
        print(f'  🔍 Scraping kategori: {category} ...')
        try:
            response = requests.get(url, headers=self.headers, timeout=10)
            response.raise_for_status()
            feed = feedparser.parse(response.content)

            articles = []
            for entry in feed.entries:
                article = {
                    'kategori':   category,
                    'judul':      entry.get('title', '').strip(),
                    'link':       entry.get('link', ''),
                    'ringkasan':  BeautifulSoup(
                                    entry.get('summary', ''), 'html.parser'
                                  ).get_text().strip()[:300],
                    'tanggal':    entry.get('published', ''),
                    'scrape_at':  datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                }
                articles.append(article)
            print(f'     ✅ {len(articles)} artikel ditemukan')
            return articles
        except Exception as e:
            print(f'     ❌ Error: {e}')
            return []

    def scrape_article_detail(self, url):
        """Scrape isi lengkap artikel dari URL."""
        try:
            resp = requests.get(url, headers=self.headers, timeout=10)
            resp.raise_for_status()
            soup = BeautifulSoup(resp.text, 'lxml')

            # Ambil konten artikel
            content_div = soup.find('div', class_='detail__body-text')
            if not content_div:
                content_div = soup.find('article')

            if content_div:
                paragraphs = content_div.find_all('p')
                content = ' '.join(p.get_text().strip() for p in paragraphs)
                return content[:1000]  # limit 1000 karakter
            return 'Konten tidak tersedia'
        except Exception as e:
            return f'Error: {e}'

    def run_all(self, feeds_dict):
        """Jalankan scraping semua kategori."""
        print('🚀 Memulai scraping berita...\n')
        for category, url in feeds_dict.items():
            articles = self.scrape_rss(category, url)
            self.all_articles.extend(articles)
            time.sleep(1)  # delay sopan
        print(f'\n📊 Total artikel terkumpul: {len(self.all_articles)}')
        return self.all_articles

    @staticmethod
    def fetch_with_retry(url, headers, retries=3):
        for i in range(retries):
            try:
                return requests.get(url, headers=headers, timeout=10)
            except:
                time.sleep(2)
        return None

    def get_tech_articles(self):
      """Filter artikel teknologi."""
      if not self.all_articles:
          print('❌ Harap jalankan metode run_all terlebih dahulu.')
          return pd.DataFrame()
      df = pd.DataFrame(self.all_articles)
      keywords = ['teknologi', 'digital', 'ai', 'internet', 'aplikasi']
      mask = df['judul'].str.lower().str.contains('|'.join(keywords), na=False)
      df_tech = df[mask]
      print(f'✅ {len(df)} artikel teknologi ditemukan')
      return df_tech

print('✅ Class NewsScraper siap digunakan!')

✅ Class NewsScraper siap digunakan!


# 4. Jalankan Scrapping

In [18]:
scraper = NewsScraper(headers=HEADERS)
all_articles = scraper.run_all(RSS_FEEDS)

#Filter teknologi dipanggil setelah data ada
df_tech = scraper.get_tech_articles()
print(df_tech.head(5).to_string())

🚀 Memulai scraping berita...

  🔍 Scraping kategori: terkini ...
     ✅ 50 artikel ditemukan
  🔍 Scraping kategori: ekonomi ...
     ✅ 20 artikel ditemukan

📊 Total artikel terkumpul: 70
✅ 70 artikel teknologi ditemukan
   kategori                                                                                            judul                                                                                                                                     link                                                                                                                  ringkasan                          tanggal            scrape_at
2   terkini                                Mendiktisaintek dorong pembentukan pengembangan satelit Indonesia                              https://www.antaranews.com/berita/5538768/mendiktisaintek-dorong-pembentukan-pengembangan-satelit-indonesia      Menteri Pendidikan Tinggi, Sains, dan Teknologi (Mendiktisaintek) Brian Yuliarto mendorong pembentukan kon

#5. Simpan ke DataFrame & Analisis

In [19]:
df = pd.DataFrame(all_articles)

print('Preview Data:')
print(df.head(5).to_string())
print(f'\n Shape: {df.shape}')
print(f'\n Jumlah Artikel: {len(df)}')
print(df['kategori'].value_counts())


Preview Data:
  kategori                                                                           judul                                                                                                                     link                                                                                                                    ringkasan                          tanggal            scrape_at
0  terkini                    Ford Mustang GTD pecahkan rekor waktu putaran di Nürburgring               https://otomotif.antaranews.com/berita/5538771/ford-mustang-gtd-pecahkan-rekor-waktu-putaran-di-nrburgring                     Produsen otomotif asal Amerika Serikat, Ford, memecahkan rekor waktu putaran kendaraannya di sirkuit ...  Thu, 23 Apr 2026 10:36:11 +0700  2026-04-23 03:37:07
1  terkini            Mentan: Stok cadangan beras pemerintah di Bulog tembus 5,19 juta ton             https://www.antaranews.com/berita/5538769/mentan-stok-cadangan-beras-pemerintah-di-bulog-tembus-519-j

In [20]:
# Cek sample detail
print('📰 Contoh Artikel Pertama:')
sample = df.iloc[0]
print(f'Judul    : {sample["judul"]}')
print(f'Kategori : {sample["kategori"]}')
print(f'Link     : {sample["link"]}')
print(f'Tanggal  : {sample["tanggal"]}')
print(f'Ringkasan: {sample["ringkasan"][:200]}...')

📰 Contoh Artikel Pertama:
Judul    : Ford Mustang GTD pecahkan rekor waktu putaran di Nürburgring
Kategori : terkini
Link     : https://otomotif.antaranews.com/berita/5538771/ford-mustang-gtd-pecahkan-rekor-waktu-putaran-di-nrburgring
Tanggal  : Thu, 23 Apr 2026 10:36:11 +0700
Ringkasan: Produsen otomotif asal Amerika Serikat, Ford, memecahkan rekor waktu putaran kendaraannya di sirkuit ......


#6. Ekspor ke CSV & Excel

In [21]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Simpan CSV
csv_file = f'news_data_{timestamp}.csv'
df.to_csv(csv_file, index=False, encoding='utf-8-sig')

# Simpan Excel
excel_file = f'news_data_{timestamp}.xlsx'
df.to_excel(excel_file, index=False)

print(f'✅ Data disimpan ke:')
print(f'   📄 {csv_file}')
print(f'   📊 {excel_file}')

✅ Data disimpan ke:
   📄 news_data_20260423_033743.csv
   📊 news_data_20260423_033743.xlsx


In [22]:
# Download file di Colab
from google.colab import files
files.download(csv_file)
print('📥 File CSV berhasil di-download!')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 File CSV berhasil di-download!
